# 02 — TensorRT INT8 量化

对应论文 **Figure 3** 与 Table 2。输入为 notebook 01 的 `weights/retrain.pt`（若缺失则回退到 `yolov8n.pt`）。

Paper Figure 3 / Table 2. INT8 min-max quantization under TensorRT.



## 量化公式（论文 10–14）

\(q = (x-\min)/(\max-\min)\times 255\)，再按 scale / zero-point 做 int8 卷积。标定默认 **min-max**（`IInt8MinMaxCalibrator`）。

需要 NVIDIA GPU + TensorRT。无 GPU 时只能导出 ONNX，跳过 engine 构建。



In [ ]:
import sys
from pathlib import Path

import torch
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.eval_utils import benchmark_ms, coco128_train_images, val_map
from src.int8_trt import build_int8_engine, export_onnx

DATA = "coco128.yaml"
IMGSZ = 640
N_CALIB = 64
WEIGHTS = ROOT / "weights"
PT = WEIGHTS / "retrain.pt"
if not PT.exists():
    PT = Path("yolov8n.pt")
    print("retrain.pt not found, fallback:", PT)

SAMPLE = "https://ultralytics.com/images/bus.jpg"
HAS_CUDA = torch.cuda.is_available()
print("pt:", PT, "| cuda:", HAS_CUDA)



## 1. 导出 ONNX



In [ ]:
onnx_path, yolo = export_onnx(PT, imgsz=IMGSZ)
print("onnx:", onnx_path, "size_mb:", round(onnx_path.stat().st_size / 1024 / 1024, 2))



## 2a. Ultralytics TensorRT INT8（简便路径）

内部走 TensorRT 标定。论文表格对应的是剪枝后模型的 INT8 engine。



In [ ]:
engine_ultra = None
if HAS_CUDA:
    try:
        yolo_trt = YOLO(str(PT))
        exported = yolo_trt.export(
            format="engine",
            int8=True,
            data=DATA,
            imgsz=IMGSZ,
            workspace=4,
            device=0,
        )
        engine_ultra = Path(exported) if exported else PT.with_suffix(".engine")
        print("ultralytics engine:", engine_ultra)
    except Exception as e:
        print("Ultralytics TensorRT export failed:", type(e).__name__, e)
else:
    print("skip: CUDA not available")



## 2b. 论文同款 min-max 标定（Polygraphy）

标定图需已经过前处理：resize、RGB、除以 255。默认使用 coco128 训练集图像。



In [ ]:
engine_minmax = None
calib_dir = coco128_train_images()
print("calib_dir:", calib_dir, "| exists:", Path(calib_dir).exists())

if HAS_CUDA and Path(calib_dir).exists():
    try:
        engine_minmax = build_int8_engine(
            onnx_path,
            yolo,
            calib_dir=calib_dir,
            engine_path=WEIGHTS / "retrain_int8_minmax.engine",
            n_images=N_CALIB,
            imgsz=IMGSZ,
            calibration_method="min-max",
        )
    except Exception as e:
        print("min-max engine failed:", type(e).__name__, e)
else:
    print("skip min-max engine (need CUDA + calib images)")



## 3. Table 2 对比

论文结果（COCO / bus.jpg）：

| | Engine size | Speed | Acc |
|---|---:|---:|---:|
| FP32 engine | 20.1 MB | 53.5 ms | 0.754 |
| INT8 engine | 7.2 MB | 205 ms | 0.722 |

压缩率约 64.2%，精度约下降 4.2%。



In [ ]:
def eval_engine(tag, path):
    path = Path(path)
    if not path.exists():
        print(tag, "missing")
        return
    m = YOLO(str(path), task="detect")
    size_mb = path.stat().st_size / 1024 / 1024
    ms = benchmark_ms(m, SAMPLE, n=20)
    metrics = val_map(m, DATA)
    print(f"{tag:18} {size_mb:6.2f} MB  {ms:7.1f} ms  mAP50={metrics['map50']:.4f}  mAP={metrics['map']:.4f}")

print(f"{'model':18} {'size':>10}  {'speed':>10}  mAP")
eval_engine("pytorch_pt", PT)
if engine_ultra:
    eval_engine("trt_int8_ultra", engine_ultra)
if engine_minmax:
    eval_engine("trt_int8_minmax", engine_minmax)

